In [5]:
import pandas as pd
import numpy as np
#
# Dataset
#
data = {
'Weather':
['Sunny','Sunny','Cloudy','Rainy','Rainy','Rainy','Cloudy','Sunny','Sunny','Rainy','Sunny','Cloudy','Cloudy','Rainy'],
'Temperature':
['Hot','Hot','Hot','Mild','Cool','Cool','Cool','Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
'SoilMoisture':
['Dry','Dry','Dry','Dry','Moist','Moist','Moist','Dry','Moist','Moist','Moist','Dry','Moist','Dry'],
'Wind':
['Weak','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Weak','Weak','Strong','Strong','Weak','Strong'],
'Irrigate': ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No']
}
df = pd.DataFrame(data)

# STEP 1: ENTROPY

def entropy(target):
  values, counts = np.unique(target, return_counts=True)
  prob = counts / counts.sum()
  return -np.sum(prob * np.log2(prob))

# STEP 2: INFORMATION GAIN

def information_gain(data, feature, target_name):
  total_entropy = entropy(data[target_name])
  values, counts = np.unique(data[feature], return_counts=True)
  weighted_entropy = 0
  for i in range(len(values)):
    subset = data[data[feature] == values[i]]
    weighted_entropy += (counts[i] / np.sum(counts)) * entropy(subset[target_name])

  return total_entropy - weighted_entropy


# STEP 3: ID3 ALGORITHM
def id3(data, features, target_name, depth=0):
  indent = " " * depth
  # If all target values are same -> return leaf
  if len(np.unique(data[target_name])) == 1:
    return np.unique(data[target_name])[0]
  # If no features left -> return majority class
  if len(features) == 0:
    return data[target_name].mode()[0]
  # Compute IG for all features
  gains = []
  print(f"\n{indent}Evaluating features at depth {depth}:")
  for feature in features:
    gain = information_gain(data, feature, target_name)
    gains.append(gain)
    print(f"{indent}IG({feature}) = {gain:.4f}")
  # Select best feature
  best_feature = features[np.argmax(gains)]
  print(f"{indent}Best Feature -> {best_feature}")
  tree = {best_feature: {}}
  # Split dataset
  for value in np.unique(data[best_feature]):
    print(f"{indent} Splitting {best_feature} = {value}")
    subset = data[data[best_feature] == value]
    if subset.shape[0] == 0:
      tree[best_feature][value] = data[target_name].mode()[0]
    else:
      remaining_features = [f for f in features if f != best_feature]
      subtree = id3(subset, remaining_features, target_name, depth + 1)
    tree[best_feature][value] = subtree
  return tree

# STEP 4: PRINT TREE FUNCTION
def print_tree(tree, indent=""):
  if not isinstance(tree, dict):
    print(indent + "->", tree)
    return
  for feature, branches in tree.items():
    for value, subtree in branches.items():
      print(f"{indent}{feature} = {value}:")
      print_tree(subtree, indent + "  ")

# STEP 5: BUILD TREE

features = list(df.columns[ :- 1])
tree_model = id3(df, features, 'Irrigate')

# STEP 6: DISPLAY TREE
print("\nFinal Decision Tree:\n")
print_tree(tree_model)



Evaluating features at depth 0:
IG(Weather) = 0.2467
IG(Temperature) = 0.0292
IG(SoilMoisture) = 0.1518
IG(Wind) = 0.0481
Best Feature -> Weather
 Splitting Weather = Cloudy
 Splitting Weather = Rainy

 Evaluating features at depth 1:
 IG(Temperature) = 0.0200
 IG(SoilMoisture) = 0.0200
 IG(Wind) = 0.9710
 Best Feature -> Wind
  Splitting Wind = Strong
  Splitting Wind = Weak
 Splitting Weather = Sunny

 Evaluating features at depth 1:
 IG(Temperature) = 0.5710
 IG(SoilMoisture) = 0.9710
 IG(Wind) = 0.0200
 Best Feature -> SoilMoisture
  Splitting SoilMoisture = Dry
  Splitting SoilMoisture = Moist

Final Decision Tree:

Weather = Cloudy:
  -> Yes
Weather = Rainy:
  Wind = Strong:
    -> No
  Wind = Weak:
    -> Yes
Weather = Sunny:
  SoilMoisture = Dry:
    -> No
  SoilMoisture = Moist:
    -> Yes
